In [2]:
import os, sys
from pathlib import Path
for _p in [Path.cwd(), Path.cwd() / "labs", *Path.cwd().parents]:
    if (_p / "common" / "__init__.py").exists():
        os.chdir(_p)
        sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError("Ordner labs/ nicht gefunden. Öffnen Sie das Notebook aus dem Ordner labs/ des Kursmaterials.")

import json
import logging
import time
import warnings
from contextlib import ExitStack

# Hinweise ohne Belang für die Demo ausblenden (TestClient, Fortschrittsbalken, Anlegen der MLflow-Datenbank)
warnings.filterwarnings("ignore", message="Using `httpx` with `starlette.testclient`")
warnings.filterwarnings("ignore", message="IProgress not found")
for noisy in ("alembic", "mlflow"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

import httpx
import mlflow
import numpy as np
import pydantic

from common.paths import offline, LABS_DIR, MODELS_DIR, ARTIFACTS_DIR

offline()

from common.data import load_banking77, dataset_hash
from common.evalset import build_evalset
from common.interactive import explore
from common.metrics import retrieval_metrics
from common.models import load_encoder, encode
from common.plotstyle import use_course_style
from common.serving import (
    LAB3_DIR, RUN_DOCKER, BackgroundStream, ProbeScan, api_quality, build_index, choose_classifier, container_env, docker,
    engine_quality, free_port, inprocess_client, make_engine, plot_pair_matrix, print_stream, probe_cosines,
    process_memory_mb, running_count, running_service, running_services, service_command, service_settings,
    show_response, wait_for_health, wait_until_interrupted, write_deployment,
)
from common.tracking import setup_mlflow
from lab3_service.engine import IndexMismatchError, check_compatibility, load_index

use_course_style()
setup_mlflow("lab3")
OUT_DIR = LAB3_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

train, test, intents = load_banking77()
evalset = build_evalset(train, test)
DATA_HASH = dataset_hash(train, test)[:16]
CORPUS_INTENTS = train["intent"].tolist()
QUERY_INTENTS = [intents[label] for label in evalset.query_labels]
BLUE, GREEN = "minilm-l6", "bge-small"  # läuft heute, Kandidat aus Lab 1
print(f"Korpus {len(evalset.corpus)} Anfragen, Eval-Set {evalset.fingerprint}, Daten-Hash {DATA_HASH}")
print(f"Docker-Zellen aktiv: {RUN_DOCKER}")

Korpus 9993 Anfragen, Eval-Set e21d87d6477c7f22, Daten-Hash 3176e3ac9aef7185
Docker-Zellen aktiv: False


In [3]:
encoder_blue = load_encoder(BLUE)
index_dir_blue = OUT_DIR / f"index_{BLUE}"
meta_blue, corpus_emb_blue, seconds = build_index(BLUE, encoder_blue, evalset, CORPUS_INTENTS, DATA_HASH, index_dir_blue)

size_mb = sum(f.stat().st_size for f in index_dir_blue.iterdir()) / 1e6
print(f"{index_dir_blue.relative_to(LABS_DIR)}: {meta_blue['size']} x {meta_blue['dim']}, {size_mb:.0f} MB, "
      f"gebaut in {seconds:.1f} s")
print((index_dir_blue / "meta.json").read_text())

artifacts/lab3/index_minilm-l6: 9993 x 384, 16 MB, gebaut in 4.1 s
{
  "model_name": "minilm-l6",
  "dim": 384,
  "size": 9993,
  "normalized": true,
  "data_hash": "3176e3ac9aef7185",
  "evalset_fingerprint": "e21d87d6477c7f22",
  "fingerprint": "66e93b278fd1d28f"
}


In [4]:
classifier = choose_classifier(OUT_DIR, evalset, intents)
settings_blue = service_settings(BLUE, index_dir_blue, classifier)
print("Klassifikator:", classifier.describe())
print("Start im Terminal (Ordner labs/):\n ", service_command(settings_blue))


Klassifikator: fallback-logreg: logistische Regression je Encoder unter artifacts/lab3/service_base
Start im Terminal (Ordner labs/):
  LAB_BASE_DIR=artifacts/lab3/service_base ENCODER_PATH=models/minilm-l6 INDEX_DIR=artifacts/lab3/index_minilm-l6 uvicorn lab3_service.app:app --port 8000


In [5]:
client = inprocess_client(settings_blue)  # Start wie im Betrieb: lifespan läuft, Modelle laden einmal
show_response(client.get("/health"), keys=("status", "encoder", "classifier", "index"))
show_response(client.post("/classify", json={"text": "I am still waiting on my card?"}),
              keys=("intent", "confidence", "source"))
show_response(client.post("/similar", json={"text": "My card was stolen, what now?", "k": 3}))

GET /health -> 200
  {"status": "ok", "encoder": {"name": "minilm-l6", "source": "env", "dim": 384}, "classifier": {"name": "logreg-on-minilm-l6", "source": "fallback-logreg", "n_intents": 77}, "index": {"model_name": "minilm-l6", "size": 9993, "dim": 384, "fingerprint": "66e93b278fd1d28f", "data_hash": "3176e3ac9aef7185"}}
POST /classify -> 200
  {"intent": "card_arrival", "confidence": 0.735, "source": "fallback-logreg"}
POST /similar -> 200
  {"model": "minilm-l6", "index_fingerprint": "66e93b278fd1d28f", "results": [{"rank": 1, "text": "My card was stolen", "intent": "lost_or_stolen_card", "score": 0.93}, {"rank": 2, "text": "My card was stolen. What do I need to do?", "intent": "lost_or_stolen_card", "score": 0.919}, {"rank": 3, "text": "What do I do if my card was stolen.", "intent": "lost_or_stolen_card", "score": 0.916}]}
